# v1 Global Temporal Transformer — train on CT-RATE cached features

End-to-end: pull the **cached frozen MERLIN features** from GCS, train the small
temporal module (the ONLY trainable part), and report F1 **per class** and **per disease**.

**Architecture (matches `docs/model_v1_global.md`):** frozen MERLIN gives two global
512-d vectors per pair (`v_prior`, `v_current`, cached as `pooled`). The trainable module =
`W` projection + role embeddings (prior/current) + a learnable `e_diff` query + a tiny
2-layer Transformer + a 512-d output head + a learnable `logit_scale`. Its output `v_d` is
compared by cosine to frozen text prototypes.

**Multi-finding handling:** a CT-RATE pair has many findings, each with its own
worsened/stable/improved label, but the module makes ONE `v_d` per pair. So each
`(pair, finding)` example is classified by cosine of the shared `v_d` against **that
finding's own 3 prototypes** (from the cached 54-entry prompt bank = 18 findings x 3
classes). Different findings -> different prototype triplets -> different predictions from
the same `v_d`. This gives per-disease F1 for free.

**Uploads you need:** `medgemma_labels_v3.jsonl` and `subset_pairs.csv` (both small).
Everything else is pulled from `gs://neal-ct-temporal-features`. No GPU required.

In [ ]:
# 1. Setup + auth
!pip -q install scikit-learn 2>/dev/null
import os, io, json, csv, numpy as np, torch, torch.nn as nn
from collections import Counter, defaultdict
from sklearn.metrics import f1_score, confusion_matrix
csv.field_size_limit(10**9)

PROJECT_ID = 'neal-ct-temporal'
BUCKET     = 'neal-ct-temporal-features'
from google.colab import auth
auth.authenticate_user()
!gcloud config set project {PROJECT_ID} -q

CLASSES = ['worsened', 'stable', 'improved']   # fixed label order (indices 0,1,2)
C2I = {c: i for i, c in enumerate(CLASSES)}
torch.manual_seed(0); np.random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
# 2. Pull cached features (pooled) + prompt bank from GCS
os.makedirs('/content/feat', exist_ok=True)
print('downloading image features (~1.3 GB, one-time)...')
!gsutil -m -q cp 'gs://{BUCKET}/features/*.npz' /content/feat/ 2>/dev/null
!gsutil -q cp 'gs://{BUCKET}/text/prompt_bank_3.npz' /content/prompt_bank_3.npz

# load pooled 512-d per volume (ignore the big grid — v1 is global)
POOLED = {}
for fn in os.listdir('/content/feat'):
    if not fn.endswith('.npz') or fn.startswith('_'):
        continue
    d = np.load(f'/content/feat/{fn}')
    POOLED[fn[:-4]] = d['pooled'].astype('float32')   # key = volume name w/o .npz
print('loaded pooled vectors for', len(POOLED), 'volumes')

# build finding-specific prototypes: proto[finding] -> (3, 512) in CLASSES order
pb = np.load('/content/prompt_bank_3.npz')
def l2(x): return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-8)
PROTO = {}
findings_in_bank = sorted({k.split('|')[0] for k in pb.files})
for f in findings_in_bank:
    PROTO[f] = l2(np.stack([pb[f'{f}|{c}'].astype('float32') for c in CLASSES]))
print('prototypes for', len(PROTO), 'findings, each', PROTO[findings_in_bank[0]].shape)

In [ ]:
# 3. Upload labels + splits, then build (pair, finding) examples per split
from google.colab import files
print('Upload BOTH: medgemma_labels_v3.jsonl  and  subset_pairs.csv')
up = files.upload()
LAB = 'medgemma_labels_v3.jsonl'
SUB = 'subset_pairs.csv'

# split per pair from subset_pairs.csv
split_of = {}
for r in csv.DictReader(open(SUB)):
    split_of[(r['patient'], r['prior_volume'], r['curr_volume'])] = r['split']

# labels keyed by pair
recs = {}
for line in open(LAB, encoding='utf-8'):
    if not line.strip():
        continue
    x = json.loads(line)
    recs[(x['patient'], x['prior_volume'], x['curr_volume'])] = x

def vkey(v): return v.replace('.nii.gz', '')

# examples: one per (pair, gold finding). skip if features or prototypes missing.
examples = {'train': [], 'val': [], 'test': []}
skipped = Counter()
for key, sp in split_of.items():
    patient, pv, cv = key
    if vkey(pv) not in POOLED or vkey(cv) not in POOLED:
        skipped['no_feature'] += 1; continue
    rec = recs.get(key)
    if rec is None:
        skipped['no_label'] += 1; continue
    for fd in rec['findings']:
        if fd.get('tier') != 'explicit':
            continue
        d = fd.get('direction')
        f = fd.get('finding')
        if d not in C2I or f not in PROTO:
            skipped['bad_dir_or_finding'] += 1; continue
        examples[sp].append({'vp': vkey(pv), 'vc': vkey(cv),
                             'finding': f, 'y': C2I[d]})

for sp in ['train', 'val', 'test']:
    cc = Counter(e['y'] for e in examples[sp])
    print(f'{sp:5}: {len(examples[sp]):5} examples  '
          f"(worsened={cc[0]} stable={cc[1]} improved={cc[2]})")
print('skipped:', dict(skipped))

In [ ]:
# 4. Tensorize each split (VP, VC, PROTO(3x512), Y, and finding name for per-disease F1)
def tensorize(exs):
    VP = torch.tensor(np.stack([POOLED[e['vp']] for e in exs]))
    VC = torch.tensor(np.stack([POOLED[e['vc']] for e in exs]))
    PR = torch.tensor(np.stack([PROTO[e['finding']] for e in exs]))   # (N,3,512)
    Y  = torch.tensor([e['y'] for e in exs], dtype=torch.long)
    F  = [e['finding'] for e in exs]
    return VP, VC, PR, Y, F

DATA = {sp: tensorize(examples[sp]) for sp in ['train', 'val', 'test']}

# inverse-frequency class weights from TRAIN
cnt = Counter(DATA['train'][3].tolist())
tot = sum(cnt.values())
W = torch.tensor([tot / (3 * cnt[i]) for i in range(3)], dtype=torch.float32)
print('class weights (w/s/i):', [round(w, 3) for w in W.tolist()])

In [ ]:
# 5. The trainable temporal module (the ONLY trained component)
class TemporalModule(nn.Module):
    def __init__(self, d_in=512, d_model=256, n_layers=2, n_heads=4,
                 dropout=0.1, antisym=False):
        super().__init__()
        self.W = nn.Linear(d_in, d_model)                 # shared input projection
        self.role = nn.Parameter(torch.randn(2, d_model) * 0.02)  # prior / current tags
        self.e_diff = nn.Parameter(torch.randn(1, d_model) * 0.02) # learnable query
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4,
                                           dropout=dropout, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model, d_in)              # back to 512-d text space
        self.logit_scale = nn.Parameter(torch.tensor(np.log(1 / 0.07)))
        self.antisym = antisym

    def _pass(self, vp, vc):
        B = vp.size(0)
        tp = self.W(vp) + self.role[0]
        tc = self.W(vc) + self.role[1]
        ed = self.e_diff.expand(B, -1)
        toks = torch.stack([ed, tp, tc], dim=1)           # (B,3,d_model)
        h = self.enc(toks)
        return self.head(h[:, 0])                          # v_d = e_diff readout (B,512)

    def forward(self, vp, vc):
        vd = self._pass(vp, vc) - self._pass(vc, vp) if self.antisym else self._pass(vp, vc)
        return vd

def logits_from(vd, proto, logit_scale):
    vd = nn.functional.normalize(vd, dim=-1)               # (B,512)
    pr = nn.functional.normalize(proto, dim=-1)            # (B,3,512)
    cos = torch.einsum('bd,bkd->bk', vd, pr)               # (B,3)
    return logit_scale.exp().clamp(max=100) * cos

n_params = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
print('module trainable params:', f'{n_params(TemporalModule()):,}')

In [ ]:
# 6. Train (early-stop on val macro-F1)
ANTISYM = False        # flip to True for the antisymmetry ablation
D_MODEL = 256
EPOCHS  = 100
LR      = 1e-3
BATCH   = 256
PATIENCE = 15

model = TemporalModule(d_model=D_MODEL, antisym=ANTISYM).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
ce = nn.CrossEntropyLoss(weight=W.to(DEVICE))

def evaluate(split):
    VP, VC, PR, Y, F = DATA[split]
    model.eval()
    with torch.no_grad():
        vd = model(VP.to(DEVICE), VC.to(DEVICE))
        lg = logits_from(vd, PR.to(DEVICE), model.logit_scale)
        pred = lg.argmax(1).cpu().numpy()
    y = Y.numpy()
    return y, pred, F

VP, VC, PR, Y, _ = DATA['train']
N = len(Y)
best_val, best_state, bad = -1, None, 0
for ep in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(N)
    tot_loss = 0.0
    for i in range(0, N, BATCH):
        idx = perm[i:i + BATCH]
        vd = model(VP[idx].to(DEVICE), VC[idx].to(DEVICE))
        lg = logits_from(vd, PR[idx].to(DEVICE), model.logit_scale)
        loss = ce(lg, Y[idx].to(DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()
        tot_loss += loss.item() * len(idx)
    yv, pv, _ = evaluate('val')
    vf1 = f1_score(yv, pv, labels=[0, 1, 2], average='macro', zero_division=0)
    if vf1 > best_val:
        best_val, best_state, bad = vf1, {k: v.detach().cpu().clone()
                                          for k, v in model.state_dict().items()}, 0
    else:
        bad += 1
    if ep % 10 == 0 or ep == 1:
        print(f'ep {ep:3}  loss {tot_loss / N:.3f}  val_macroF1 {vf1:.3f}  (best {best_val:.3f})')
    if bad >= PATIENCE:
        print(f'early stop at ep {ep} (best val macro-F1 {best_val:.3f})'); break

model.load_state_dict(best_state)
print('restored best model. val macro-F1 =', round(best_val, 3))

In [ ]:
# 7. TEST metrics — per class
y, pred, F = evaluate('test')
macro = f1_score(y, pred, labels=[0, 1, 2], average='macro', zero_division=0)
percls = f1_score(y, pred, labels=[0, 1, 2], average=None, zero_division=0)
acc = (y == pred).mean()

# reference: always predict majority class (from train)
maj = Counter(DATA['train'][3].tolist()).most_common(1)[0][0]
maj_macro = f1_score(y, np.full_like(y, maj), labels=[0, 1, 2], average='macro', zero_division=0)

print('=== TEST — per class ===')
print(f'accuracy        : {acc:.3f}')
print(f'macro-F1        : {macro:.3f}   (always-{CLASSES[maj]} ref = {maj_macro:.3f})')
for i, c in enumerate(CLASSES):
    print(f'  F1 {c:9}: {percls[i]:.3f}')
print('\nconfusion matrix (rows=true, cols=pred; order w/s/i):')
print(confusion_matrix(y, pred, labels=[0, 1, 2]))

=== TEST — per class ===
accuracy        : 0.401
macro-F1        : 0.398   (always-worsened ref = 0.206)
  F1 worsened : 0.406
  F1 stable   : 0.361
  F1 improved : 0.428

confusion matrix (rows=true, cols=pred; order w/s/i):
[[38 29 46]
 [23 26 26]
 [13 14 37]]


In [ ]:
# 8. TEST metrics — per disease (finding)
by_f = defaultdict(lambda: {'y': [], 'p': []})
for yi, pi, fi in zip(y, pred, F):
    by_f[fi]['y'].append(yi); by_f[fi]['p'].append(pi)

rows = []
for f, d in by_f.items():
    yy, pp = np.array(d['y']), np.array(d['p'])
    present = sorted(set(yy.tolist()))     # classes actually present for this finding
    mf1 = f1_score(yy, pp, labels=present, average='macro', zero_division=0)
    rows.append((f, len(yy), (yy == pp).mean(), mf1))

rows.sort(key=lambda r: -r[1])
print('=== TEST — per disease ===')
print(f"{'finding':<34}{'n':>5}{'acc':>7}{'macroF1*':>10}")
for f, n, a, mf1 in rows:
    print(f'{f:<34}{n:>5}{a:>7.3f}{mf1:>10.3f}')
print('\n* macro-F1 over the classes actually present for that finding')
print('  (many chronic findings never "improve", so their support is 2 classes).')

=== TEST — per disease ===
finding                               n    acc  macroF1*
Pericardial effusion                 26  0.385     0.185
Cardiomegaly                         23  0.435     0.330
Pleural effusion                     23  0.478     0.216
Lymphadenopathy                      19  0.421     0.198
Consolidation                        18  0.278     0.152
Lung nodule                          18  0.389     0.187
Peribronchial thickening             16  0.188     0.125
Lung opacity                         15  0.400     0.200
Pulmonary fibrotic sequela           13  0.231     0.125
Medical material                     13  0.231     0.214
Emphysema                            12  0.583     0.386
Atelectasis                          12  0.833     0.633
Bronchiectasis                       10  0.500     0.522
Interlobular septal thickening       10  0.400     0.315
Arterial wall calcification           8  0.625     0.385
Mosaic attenuation pattern            6  0.167     0.111
Coro

In [ ]:
# 9. NAIVE zero-shot baseline — d = v_current − v_prior, cosine vs finding prompts.
#    No transformer, no logit_scale, no training. Pure retrieval floor.
import torch, numpy as np
from sklearn.metrics import f1_score, confusion_matrix
from collections import Counter, defaultdict

def zeroshot_eval(split):
    VP, VC, PR, Y, F = DATA[split]
    d  = torch.nn.functional.normalize(VC - VP, dim=-1)        # (N,512) difference embedding
    pr = torch.nn.functional.normalize(PR, dim=-1)             # (N,3,512) finding prototypes
    cos = torch.einsum('bd,bkd->bk', d, pr)                    # (N,3) cosine to w/s/i prompts
    pred = cos.argmax(1).numpy()
    return Y.numpy(), pred, F

y, pred, F = zeroshot_eval('test')
macro  = f1_score(y, pred, labels=[0,1,2], average='macro',    zero_division=0)
percls = f1_score(y, pred, labels=[0,1,2], average=None,       zero_division=0)
acc    = (y == pred).mean()
maj    = Counter(DATA['train'][3].tolist()).most_common(1)[0][0]
maj_f1 = f1_score(y, np.full_like(y, maj), labels=[0,1,2], average='macro', zero_division=0)

print('=== NAIVE zero-shot (d = v_cur - v_pri, cosine) - per class ===')
print(f'accuracy   : {acc:.3f}')
print(f'macro-F1   : {macro:.3f}   (always-{CLASSES[maj]} ref = {maj_f1:.3f})')
for i, c in enumerate(CLASSES):
    print(f'  F1 {c:9}: {percls[i]:.3f}')
print('confusion (rows=true, cols=pred; w/s/i):')
print(confusion_matrix(y, pred, labels=[0,1,2]))

# per-disease (same format as the trained model's cell 8)
by = defaultdict(lambda: {'y': [], 'p': []})
for yi, pi, fi in zip(y, pred, F):
    by[fi]['y'].append(yi); by[fi]['p'].append(pi)
print('\n=== NAIVE zero-shot - per disease ===')
print(f"{'finding':<34}{'n':>5}{'acc':>7}{'macroF1*':>10}")
for f, dd in sorted(by.items(), key=lambda kv: -len(kv[1]['y'])):
    yy, pp = np.array(dd['y']), np.array(dd['p']); present = sorted(set(yy.tolist()))
    print(f"{f:<34}{len(yy):>5}{(yy==pp).mean():>7.3f}"
          f"{f1_score(yy,pp,labels=present,average='macro',zero_division=0):>10.3f}")
print('\nNote: d is a difference of IMAGE embeddings matched to ABSOLUTE-state text')
print('prompts (cross-space) - this is the naive floor the trained module should beat.')


=== NAIVE zero-shot (d = v_cur - v_pri, cosine) - per class ===
accuracy   : 0.365
macro-F1   : 0.352   (always-worsened ref = 0.206)
  F1 worsened : 0.429
  F1 stable   : 0.339
  F1 improved : 0.288
confusion (rows=true, cols=pred; w/s/i):
[[45 50 18]
 [31 31 13]
 [21 27 16]]

=== NAIVE zero-shot - per disease ===
finding                               n    acc  macroF1*
Pericardial effusion                 26  0.269     0.278
Cardiomegaly                         23  0.261     0.257
Pleural effusion                     23  0.391     0.391
Lymphadenopathy                      19  0.421     0.426
Consolidation                        18  0.667     0.605
Lung nodule                          18  0.278     0.245
Peribronchial thickening             16  0.438     0.452
Lung opacity                         15  0.533     0.472
Pulmonary fibrotic sequela           13  0.231     0.125
Medical material                     13  0.385     0.431
Emphysema                            12  0.167     0.127

In [ ]:
# Cosine separability of the text prompts (worsened / stable / improved)
import numpy as np
findings = sorted(PROTO.keys())
V = np.stack([PROTO[f] for f in findings])          # (F,3,512), unit-norm rows
pairs = [('worsened','stable',0,1), ('worsened','improved',0,2), ('stable','improved',1,2)]

print('=== within-finding class-prompt cosine (mean over findings) ===')
for a,b,i,j in pairs:
    v = (V[:,i,:]*V[:,j,:]).sum(-1)
    print(f'  cos({a:8},{b:8}) = {v.mean():.3f}   [min {v.min():.3f}, max {v.max():.3f}]')
offdiag = np.mean([(V[:,i,:]*V[:,j,:]).sum(-1).mean() for _,_,i,j in pairs])
print(f'  mean class off-diagonal: {offdiag:.3f}   (->1.0 = classes nearly identical = unseparable)')

# worsened vs improved: is there a usable DIRECTION axis?
wi = (V[:,0,:]*V[:,2,:]).sum(-1)
print(f'\nworsened<->improved cosine: mean {wi.mean():.3f}')
print('  (lower/negative = prompts point OPPOSITE ways, which a direction signal needs)')

# does the text space separate by DISEASE or by DIRECTION?
same_find_diff_cls = offdiag
F = len(findings)
diff_find_same_cls = np.mean([(V[:,c,:]@V[:,c,:].T)[~np.eye(F,dtype=bool)].mean() for c in range(3)])
print('\n=== what dominates the text space? ===')
print(f'  same finding, different class : {same_find_diff_cls:.3f}')
print(f'  different finding, same class : {diff_find_same_cls:.3f}')
print('  -> whichever is HIGHER is what MERLIN text encodes more strongly.')
print('     If disease > direction, cosine-to-prompts can barely read change (expected).')

# per-finding ranking (lower = its 3 classes are more distinct)
print('\n=== per-finding class separability (lower = more separable) ===')
for f, m in sorted(((f, np.mean([(V[k,i,:]*V[k,j,:]).sum() for _,_,i,j in pairs]))
                    for k, f in enumerate(findings)), key=lambda r: r[1]):
    print(f'  {f:<34} {m:.3f}')


=== within-finding class-prompt cosine (mean over findings) ===
  cos(worsened,stable  ) = 0.933   [min 0.842, max 0.977]
  cos(worsened,improved) = 0.940   [min 0.836, max 0.981]
  cos(stable  ,improved) = 0.927   [min 0.846, max 0.974]
  mean class off-diagonal: 0.933   (->1.0 = classes nearly identical = unseparable)

worsened<->improved cosine: mean 0.940
  (lower/negative = prompts point OPPOSITE ways, which a direction signal needs)

=== what dominates the text space? ===
  same finding, different class : 0.933
  different finding, same class : 0.654
  -> whichever is HIGHER is what MERLIN text encodes more strongly.
     If disease > direction, cosine-to-prompts can barely read change (expected).

=== per-finding class separability (lower = more separable) ===
  Medical material                   0.885
  Pleural effusion                   0.890
  Lymphadenopathy                    0.914
  Consolidation                      0.921
  Atelectasis                        0.922
  Lung 

## Notes

- **What trained:** only the module in cell 5 (~few M params). MERLIN encoders + text
  prototypes are frozen (features/prototypes were pre-cached).
- **Ablations:** set `ANTISYM = True` in cell 6 for the antisymmetric readout
  (`v_d = g(c,p) − g(p,c)`, so "no change" = 0 by construction); vary `D_MODEL`.
- **Expectation:** per the v1 doc, the global-pool version tends to hover near the
  always-stable floor on macro-F1 and struggle on worsened/improved. That's the
  motivation for **v2 patch-wise** (use the cached `grid`, 490×2048). This notebook is
  the clean, balanced-data restatement of v1 on CT-RATE with real per-disease numbers.
- **If you'd rather use real reports** than the 3 prototypes: swap `logits_from` for an
  InfoNCE against the pair's cached dynamic sentences (needs the report-text cache).